# Nasdaq ITCH Feed Handler v1

In [ ]:
# imports

from pynq import Overlay, allocate, MMIO
import numpy as np
import time
import gzip
import struct
import math

In [ ]:
# Constants

DMA_DATA    = 0x4040
GPIO0       = 0x4120
GPIO1       = 0x4121
GPIO2       = 0x4122    
PATH        = "/home/xilinx/jupyter_notebooks/Nasdaq/name.bit"
DATA_PATH   = "/home/xilinx/jupyter_notebooks/Nasdaq/12302019.NASDAQ_ITCH50.gz"

START_BYTES = 2


In [ ]:
# Overlay Load

ol = Overlay(PATH)
ol.download()
print("Overlay Loaded:")
dma = ol.axi_dma_0
gpio_bid   = ol.axi_gpio_0
gpio_ask   = ol.axi_gpio_1


In [ ]:
send_buffer = allocate(shape=(16,), dtype=np.uint32) 

TARGET_SYMBOL   = b"AAPL    "
target_locate   = None

# Trackers

msg_count = 0
target_msg_count = 0
last_bid_price, last_ask_price = 0, 0
last_bid_shares, last_ask_shares = 0, 0

# opening .gz file from nasdaq itch history website
with gzip.open(DATA_PATH, "rb") as data:
    while True:
        start_msg = data.read(2) # Reading the 2-byte length header
        if not start_msg: break
            
        ins_len = int.from_bytes(start_msg, byteorder='big')
        output_data = data.read(ins_len)
        if len(output_data) < ins_len: break
            
        msg_count += 1
        msg_type = output_data[0:1]

        if msg_type == b'R' and output_data[11:15] == TARGET_SYMBOL[0:4]:
            target_locate = output_data[1:3]
            print(f"Found Target! Locate: {target_locate.hex()}")

        # Message modify step for Hardware
        modified_msg = bytearray(output_data)
        keep_message = False

        if msg_type == b'S': keep_message = True
        elif target_locate and output_data[1:3] == target_locate:
            modified_msg[1:3] = (1).to_bytes(2, byteorder='big')
            # Price Division for hardware use
            offset = 32 if msg_type in [b'A', b'F', b'C'] else (31 if msg_type == b'U' else None)
            if offset:
                price = int.from_bytes(modified_msg[offset:offset+4], 'big') // 100
                modified_msg[offset:offset+4] = price.to_bytes(4, 'big')
            keep_message = True

        if keep_message:
            # Pad to 4-byte boundary (due to 32 bit packet width - change to 8 if using 64 bits)
            pad_len = math.ceil(ins_len / 4) * 4
            padded = modified_msg.ljust(pad_len, b'\x00')
            
            # Big Endian swap for nasdaq purposes
            temp_arr = np.frombuffer(padded, dtype=np.uint32)
            send_buffer[:len(temp_arr)] = temp_arr.byteswap()
            
            dma.sendchannel.transfer(send_buffer, nbytes=pad_len)
            dma.sendchannel.wait()

            # may remove this
            if msg_count % 5000 == 0:
                print(f"Processed {msg_count} msgs. AAPL Bid: {last_bid_shares}  at price:  {last_bid_price}")
                print(f"Processed {msg_count} msgs. AAPL Ask: {last_ask_shares}  at price:  {last_ask_price}")


            if msg_type != b'S':
                bid_price = gpio_bid.channel1.read()
                ask_price = gpio_ask.channel1.read()
                if bid_price != last_bid_price or ask_price != last_ask_price:
                    last_bid_price, last_ask_price = bid_price, ask_price
                    last_bid_shares = gpio_bid.channel2.read()
                    last_ask_shares = gpio_ask.channel2.read()
                    print(f"Output: {last_bid_shares} Bid shares at price: {last_bid_price} | {last_ask_shares} sk shares at price: {last_ask_price}")